In [32]:
%pip install pandas numpy matplotlib scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [33]:
import os
import sys
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

In [34]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from src.preprocess import (
    load_data,
    clean_data,
    encode_data,
    split_features_target
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [35]:
df = load_data("../data/agaricus-lepiota.data")

In [36]:
column_names = [
    "poisonous",
    "cap_shape",
    "cap_surface",
    "cap_color",
    "bruises",
    "odor",
    "gill_attachment",
    "gill_spacing",
    "gill_size",
    "gill_color",
    "stalk_shape",
    "stalk_root",
    "stalk_surface_above_ring",
    "stalk_surface_below_ring",
    "stalk_color_above_ring",
    "stalk_color_below_ring",
    "veil_type",
    "veil_color",
    "ring_number",
    "ring_type",
    "spore_print_color",
    "population",
    "habitat"
]

df.columns = column_names

In [37]:
df = clean_data(df)

In [38]:
(df == "?").sum()

poisonous                   0
cap_shape                   0
cap_surface                 0
cap_color                   0
bruises                     0
odor                        0
gill_attachment             0
gill_spacing                0
gill_size                   0
gill_color                  0
stalk_shape                 0
stalk_root                  0
stalk_surface_above_ring    0
stalk_surface_below_ring    0
stalk_color_above_ring      0
stalk_color_below_ring      0
veil_type                   0
veil_color                  0
ring_number                 0
ring_type                   0
spore_print_color           0
population                  0
habitat                     0
dtype: int64

In [39]:
df, encoders = encode_data(df)

In [40]:
X, y = split_features_target(df)

In [41]:
mi = mutual_info_classif(
    X,
    y,
    random_state=42
)

mi_df = pd.DataFrame({
    "Feature": X.columns,
    "Mutual Information": mi
})

mi_df = mi_df.sort_values(
    by="Mutual Information",
    ascending=False
)

mi_df

,Feature,Mutual Information
4,odor,0.629530
19,spore_print_color,0.330191
8,gill_color,0.283913
18,ring_type,0.218053
11,stalk_surface_above_ring,0.203732
12,stalk_surface_below_ring,0.187273
13,stalk_color_above_ring,0.180185
14,stalk_color_below_ring,0.166082
7,gill_size,0.155905
20,population,0.144846


In [42]:
""" có X và y từ file '03_Feature_Engineering.ipynb'
chia 20% dữ liệu để làm tập Test còn lại 80% là tập Temp
"""
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

""" chia tiếp tập Temp thành Train và Validatoin
25% của 80% là 20% của toàn bộ dữ liệu ban đầu
-> Train là 60%, Val là 20%
"""
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.25,
    random_state=42
)
# In số lượng để ktr
print("Số lượng mẫu tập Train:", X_train.shape[0])
print("Số lượng mẫu tập Validation (Thử):", X_val.shape[0])
print("Số lượng mẫu tập Test (Thật):", X_test.shape[0])

Số lượng mẫu tập Train: 4874
Số lượng mẫu tập Validation (Thử): 1625
Số lượng mẫu tập Test (Thật): 1625


In [43]:
print("Tinh chỉnh mô hình Decision Tree")

tham_so_depth_tot_nhat = 0
diem_tree_cao_nhat = 0

# Thử nghiệm 4 mức độ sâu khác nhau của cây
danh_sach_depth = [3, 5, 7, 10]

for depth in danh_sach_depth:
    # Khởi tạo mô hình với độ sâu hiện tại
    mo_hinh_tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    
    # Học trên tập Train
    mo_hinh_tree.fit(X_train, y_train)
    
    # Làm bài thi thử
    du_doan_val = mo_hinh_tree.predict(X_val)
    
    # Chấm điểm bài thi thử
    diem_hien_tai = accuracy_score(y_val, du_doan_val)
    print("Độ sâu cây =", depth, "-> Điểm thi thử:", diem_hien_tai)
    
    # Lưu lại thông tin nếu điểm cao hơn kỷ lục cũ
    if diem_hien_tai > diem_tree_cao_nhat:
        diem_tree_cao_nhat = diem_hien_tai
        tham_so_depth_tot_nhat = depth

print("Chọn độ sâu tốt nhất là:", tham_so_depth_tot_nhat)

Tinh chỉnh mô hình Decision Tree
Độ sâu cây = 3 -> Điểm thi thử: 0.9507692307692308
Độ sâu cây = 5 -> Điểm thi thử: 0.9803076923076923
Độ sâu cây = 7 -> Điểm thi thử: 0.9926153846153846
Độ sâu cây = 10 -> Điểm thi thử: 1.0
Chọn độ sâu tốt nhất là: 10


In [44]:
print("Tinh chỉnh mô hình Random Forest")

tham_so_so_cay_tot_nhat = 0
diem_rf_cao_nhat = 0

# Thử nghiệm 3 mức số lượng cây khác nhau
danh_sach_so_cay = [10, 50, 100]

for so_cay in danh_sach_so_cay:
    # Khởi tạo rừng với số cây hiện tại
    mo_hinh_rf = RandomForestClassifier(n_estimators=so_cay, random_state=42)
    
    # Học trên tập Train
    mo_hinh_rf.fit(X_train, y_train)
    
    # Làm bài thi thử (Validation)
    du_doan_val = mo_hinh_rf.predict(X_val)
    
    # Chấm điểm bài thi thử
    diem_hien_tai = accuracy_score(y_val, du_doan_val)
    print("Số lượng cây =", so_cay, "-> Điểm thi thử:", diem_hien_tai)
    
    # Lưu lại thông tin nếu điểm cao hơn kỷ lục cũ
    if diem_hien_tai > diem_rf_cao_nhat:
        diem_rf_cao_nhat = diem_hien_tai
        tham_so_so_cay_tot_nhat = so_cay

print("Chọn số lượng cây tốt nhất là:", tham_so_so_cay_tot_nhat)

Tinh chỉnh mô hình Random Forest
Số lượng cây = 10 -> Điểm thi thử: 1.0
Số lượng cây = 50 -> Điểm thi thử: 1.0
Số lượng cây = 100 -> Điểm thi thử: 1.0
Chọn số lượng cây tốt nhất là: 10


In [45]:
print("Đánh giá cuối cùng trên tập Test")

# 1. Tạo ra 2 mô hình với các tham số tốt nhất vừa tìm được
final_tree = DecisionTreeClassifier(max_depth=tham_so_depth_tot_nhat, random_state=42)
final_rf = RandomForestClassifier(n_estimators=tham_so_so_cay_tot_nhat, random_state=42)

# 2. Huấn luyện lại trên tập Train
final_tree.fit(X_train, y_train)
final_rf.fit(X_train, y_train)

# 3. Chấm điểm trên tập Test
ket_qua_tree = final_tree.predict(X_test)
diem_test_tree = accuracy_score(y_test, ket_qua_tree)

ket_qua_rf = final_rf.predict(X_test)
diem_test_rf = accuracy_score(y_test, ket_qua_rf)

# 4. In kết quả để so sánh
print(f"Điểm thi thật của Decision Tree : {diem_test_tree:.4f}")
print(f"Điểm thi thật của Random Forest : {diem_test_rf:.4f}")

Đánh giá cuối cùng trên tập Test
Điểm thi thật của Decision Tree : 1.0000
Điểm thi thật của Random Forest : 1.0000
